# Project: Homework Tutor

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will build a homework tutor system where a triage agent routes student questions to specialist tutors (Math Tutor and History Tutor) via handoffs.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define the Specialist Agents

Each tutor has focused instructions that shape its teaching style.

In [ ]:
from agents import Agent, Runner

math_tutor = Agent(
    name="Math Tutor",
    instructions="""You are a patient, encouraging math tutor. When a student asks a math question:
1. Identify what concept is involved
2. Explain the concept briefly
3. Walk through the solution step by step
4. Give a similar practice problem at the end

Use simple language and avoid jargon. Always show your work.""",
)

history_tutor = Agent(
    name="History Tutor",
    instructions="""You are an engaging history tutor. When a student asks a history question:
1. Provide the key facts and dates
2. Explain the historical context — what was happening before and after
3. Discuss why this event or person matters
4. Connect it to broader themes or modern relevance

Make history come alive with vivid details and storytelling.""",
)

print(f"Created: {math_tutor.name}")
print(f"Created: {history_tutor.name}")

## 4. Create the Triage Agent

The triage agent inspects each question and hands off to the appropriate tutor.

In [ ]:
triage_agent = Agent(
    name="Triage Agent",
    instructions="""You are a homework help router. Your job is to read the student's question and hand off to the right tutor:

- Math Tutor: for math problems, equations, geometry, algebra, calculus, statistics
- History Tutor: for questions about historical events, people, dates, civilizations

If the question doesn't fit either category, answer it yourself to the best of your ability.""",
    handoffs=[math_tutor, history_tutor],
)

print(f"Triage agent ready with handoffs: {[a.name for a in triage_agent.handoffs]}")

## 5. Test with a Math Question

In [ ]:
result = Runner.run_sync(triage_agent, "What is the derivative of x^3 + 2x?")

print(f"Routed to: {result.last_agent.name}")
print(f"\nAnswer:\n{result.final_output}")

## 6. Test with a History Question

In [ ]:
result = Runner.run_sync(triage_agent, "Why did the Roman Empire fall?")

print(f"Routed to: {result.last_agent.name}")
print(f"\nAnswer:\n{result.final_output}")

## 7. Run Multiple Questions and Verify Routing

In [ ]:
questions = [
    "Solve for x: 3x - 7 = 14",
    "Who was Cleopatra and why is she famous?",
    "What is the area of a circle with radius 5?",
    "What caused World War 1?",
]

for question in questions:
    result = Runner.run_sync(triage_agent, question)
    print(f"Q: {question}")
    print(f"Routed to: {result.last_agent.name}")
    print("-" * 50)

## 8. Inspect the Handoff Trace

Look at the full trace to see the triage agent's reasoning and the handoff event.

In [ ]:
result = Runner.run_sync(
    triage_agent, "What was the significance of the Battle of Thermopylae?"
)

print(f"Final agent: {result.last_agent.name}")
print(f"\nFull trace ({len(result.new_items)} items):")
for item in result.new_items:
    print(item)

## Key Takeaways

- A triage agent with `handoffs` creates a simple, powerful routing system
- Specialist agents with focused instructions produce better answers than a single generalist
- `result.last_agent.name` confirms which specialist handled the question
- This pattern scales — add more tutors (Science, English, etc.) by expanding the handoffs list